In [ ]:
%pip install tensorflow -U scikit-learn numpy

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [ ]:
(X_train, y_train), (X_test, y_test) = datasets.cifar10.load_data()
X_train_res = preprocess_input(X_train)
X_test_res = preprocess_input(X_test)
X_train, X_test = X_train / 255.0, X_test / 255.0

In [ ]:
# cnn = models.Sequential([
#     layers.Input(shape=(32, 32, 3)),
#     layers.RandomFlip('horizontal'),
    
#     layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),
    
#     layers.Flatten(),
#     layers.Dense(64, activation='relu'),
#     layers.Dropout(0.2),
#     layers.Dense(10, activation='softmax')
# ])
# cnn.compile(optimizer='adam',
#               loss='sparse_categorical_crossentropy',
#               metrics=['accuracy'])
# cnn.fit(X_train, y_train, epochs=10)
# cnn.evaluate(X_test, y_test)

# y_pred = np.argmax(cnn.predict(X_test), axis=1)


# print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
# print(f"Precision: {precision_score(y_test, y_pred, average="macro"):.4f}")
# print(f"Recall: {recall_score(y_test, y_pred, average="macro"):.4f}")
# print(f"F1-Score: {f1_score(y_test, y_pred, average="macro"):.4f}")

In [ ]:
#ResNet 1
resnet_model_1 = ResNet50(weights='imagenet',
                        include_top=False,
                        input_shape=(32, 32, 3))

resnet_model_1.trainable = False

model_1 = models.Sequential([
    resnet_model_1,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model_1.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_1.fit(X_train_res, y_train, batch_size=64, epochs=10, validation_split=0.1)

resnet_model_1.trainable = True
fine_tuning = 143

for layer in resnet_model_1.layers[:fine_tuning]:
    layer.trainable = False
    
model_1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_1.fit(X_train_res, y_train, batch_size=32, epochs=5, validation_split=0.1)
model_1.evaluate(X_test_res, y_test)

y_pred_resnet = np.argmax(model_1.predict(X_test_res), axis=1)

print(f"Accuracy: {accuracy_score(y_test, y_pred_resnet):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_resnet, average="macro"):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_resnet, average="macro"):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_resnet, average="macro"):.4f}")

Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 49s 64ms/step - accuracy: 0.4416 - loss: 2.0077 - val_accuracy: 0.5766 - val_loss: 1.3078
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 44s 63ms/step - accuracy: 0.5457 - loss: 1.4408 - val_accuracy: 0.6050 - val_loss: 1.1780
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.5815 - loss: 1.2668 - val_accuracy: 0.6226 - val_loss: 1.1123
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 42s 59ms/step - accuracy: 0.6039 - loss: 1.1705 - val_accuracy: 0.6230 - val_loss: 1.0860
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 42s 59ms/step - accuracy: 0.6218 - loss: 1.1018 - val_accuracy: 0.6394 - val_loss: 1.0546
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 42s 59ms/step - accuracy: 0.6373 - loss: 1.0540 - val_accuracy: 0.6418 - val_loss: 1.0390
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 44s 62ms/step - accuracy: 0.6496 - loss: 1.0140 - val_accuracy: 0.6468 - val_loss: 1.0193
Epoch 8/10
149/704 ━━━━━━━━━━━━━━━━━━━━ 31s 56ms/step - accuracy: 0.6663 - loss: 0.9651

In [ ]:
#ResNet 2 Changed size to 64x64
resnet_model_2 = ResNet50(weights='imagenet',
                        include_top=False,
                        input_shape=(64, 64, 3))

resnet_model_2.trainable = False

model_2 = models.Sequential([
    layers.UpSampling2D(size(2, 2))
    resnet_model_2,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model_2.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_2.fit(X_train_res, y_train, batch_size=64, epochs=10, validation_split=0.1)

resnet_model_2.trainable = True
fine_tuning = 143

for layer in resnet_model_2.layers[:fine_tuning]:
    layer.trainable = False
    
model_2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_2.fit(X_train_res, y_train, batch_size=32, epochs=5, validation_split=0.1)
model_2.evaluate(X_test_res, y_test)

y_pred_resnet = np.argmax(model_2.predict(X_test_res), axis=1)

print(f"Accuracy: {accuracy_score(y_test, y_pred_resnet):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_resnet, average="macro"):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_resnet, average="macro"):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_resnet, average="macro"):.4f}")

In [ ]:
#ResNet 3 Size 64x64 and conv4+5
resnet_model_3 = ResNet50(weights='imagenet',
                        include_top=False,
                        input_shape=(64, 64, 3))

resnet_model_3.trainable = False

model_3 = models.Sequential([
    layers.UpSampling2D(size(2, 2))
    resnet_model_3,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model_3.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_3.fit(X_train_res, y_train, batch_size=64, epochs=10, validation_split=0.1)

resnet_model_3.trainable = True
fine_tuning = 81

for layer in resnet_model_3.layers[:fine_tuning]:
    layer.trainable = False
    
model_3.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_3.fit(X_train_res, y_train, batch_size=32, epochs=5, validation_split=0.1)
model_3.evaluate(X_test_res, y_test)

y_pred_resnet = np.argmax(model_3.predict(X_test_res), axis=1)

print(f"Accuracy: {accuracy_score(y_test, y_pred_resnet):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_resnet, average="macro"):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_resnet, average="macro"):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_resnet, average="macro"):.4f}")

In [ ]:
#ResNet 4 Changed size to 128x128
resnet_model_4 = ResNet50(weights='imagenet',
                        include_top=False,
                        input_shape=(64, 64, 3))

resnet_model_4.trainable = False

model_4 = models.Sequential([
    layers.UpSampling2D(size(2, 2))
    resnet_model_4,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model_4.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_4.fit(X_train_res, y_train, batch_size=64, epochs=10, validation_split=0.1)

resnet_model_4.trainable = True
fine_tuning = 143

for layer in resnet_model_4.layers[:fine_tuning]:
    layer.trainable = False
    
model_4.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_4.fit(X_train_res, y_train, batch_size=32, epochs=5, validation_split=0.1)
model_4.evaluate(X_test_res, y_test)

y_pred_resnet = np.argmax(model_4.predict(X_test_res), axis=1)

print(f"Accuracy: {accuracy_score(y_test, y_pred_resnet):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_resnet, average="macro"):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_resnet, average="macro"):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_resnet, average="macro"):.4f}")